<a href="https://colab.research.google.com/github/christinengalle19-collab/Christine-Tsafong-BIA-PROJECTS/blob/main/Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Project 3: Machine Learning for Predicting Trading Signals

##In this project, I used domain knowledge from technical analysis to engineer features based on MACD and RSI and to define rule based Buy, Sell, and Hold signals as ground truth labels. I then trained three classification models, Logistic Regression, Random Forest, and SVM, to learn these signals from historical price datasets.  I use cross validation to evaluate the model.

##

In [ ]:
#import library
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#dowload the data from drive
!cp /content/drive/MyDrive/final_cleaned_data.csv /content/
!cp /content/drive/MyDrive/train.csv /content/
!cp /content/drive/MyDrive/val.csv /content/
!cp /content/drive/MyDrive/test.csv /content/

In [ ]:
#Load the dataset
df = pd.read_csv('/content/drive/MyDrive/final_cleaned_data.csv')
df = df.sort_index()



In [ ]:
df['EMA12'] = df['close'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['close'].ewm(span=26, adjust=False).mean()

df['MACD'] = df['EMA12'] - df['EMA26']
df['Signal_Line'] = df['MACD'].ewm(span=9, adjust=False).mean()


In [ ]:
##sell logic
df['MACD_signal'] = 0

df.loc[df['MACD'] > df['Signal_Line'], 'MACD_signal'] = 1
df.loc[df['MACD'] < df['Signal_Line'], 'MACD_signal'] = -1  #


##RSI=100-100/(1+RS)RSI=100-100/(1+RS)

In [ ]:
delta = df['close'].diff()

gain = (delta.where(delta > 0, 0))
loss = (-delta.where(delta < 0, 0))

avg_gain = gain.ewm(span=14, adjust=False).mean()
avg_loss = loss.ewm(span=14, adjust=False).mean()

rs = avg_gain / avg_loss

df['RSI'] = 100 - (100 / (1 + rs))


In [ ]:
##RSI Signals:
df['RSI_signal'] = 0

df.loc[df['RSI'] < 30, 'RSI_signal'] = 1   # Buy
df.loc[df['RSI'] > 70, 'RSI_signal'] = -1  # Sell


In [ ]:
#Final Trading Signal
df['Final_Signal'] = 0

df.loc[(df['MACD_signal'] == 1) & (df['RSI_signal'] == 1), 'Final_Signal'] = 1
df.loc[(df['MACD_signal'] == -1) & (df['RSI_signal'] == -1), 'Final_Signal'] = -1


In [ ]:
# Data Preparation
df.dropna(inplace=True)

X = df[['MACD', 'Signal_Line', 'RSI']]
y = df['Final_Signal']


In [ ]:
# Helper function to apply feature engineering
def apply_feature_engineering(df_raw):
    df_processed = df_raw.copy()
    # EMA
    df_processed['EMA12'] = df_processed['close'].ewm(span=12, adjust=False).mean()
    df_processed['EMA26'] = df_processed['close'].ewm(span=26, adjust=False).mean()

    # MACD
    df_processed['MACD'] = df_processed['EMA12'] - df_processed['EMA26']
    df_processed['Signal_Line'] = df_processed['MACD'].ewm(span=9, adjust=False).mean()

    # MACD Signal
    df_processed['MACD_signal'] = 0
    df_processed.loc[df_processed['MACD'] > df_processed['Signal_Line'], 'MACD_signal'] = 1
    df_processed.loc[df_processed['MACD'] < df_processed['Signal_Line'], 'MACD_signal'] = -1

    # RSI
    delta = df_processed['close'].diff()
    gain = (delta.where(delta > 0, 0))
    loss = (-delta.where(delta < 0, 0))
    avg_gain = gain.ewm(span=14, adjust=False).mean()
    avg_loss = loss.ewm(span=14, adjust=False).mean()
    rs = avg_gain / avg_loss
    df_processed['RSI'] = 100 - (100 / (1 + rs))

    # RSI Signal
    df_processed['RSI_signal'] = 0
    df_processed.loc[df_processed['RSI'] < 30, 'RSI_signal'] = 1
    df_processed.loc[df_processed['RSI'] > 70, 'RSI_signal'] = -1

    # Final Trading Signal
    df_processed['Final_Signal'] = 0
    df_processed.loc[(df_processed['MACD_signal'] == 1) & (df_processed['RSI_signal'] == 1), 'Final_Signal'] = 1
    df_processed.loc[(df_processed['MACD_signal'] == -1) & (df_processed['RSI_signal'] == -1), 'Final_Signal'] = -1

    df_processed.dropna(inplace=True)
    return df_processed

train = pd.read_csv('/content/train.csv')
val = pd.read_csv('/content/val.csv')
test = pd.read_csv('/content/test.csv')

train_processed = apply_feature_engineering(train)
val_processed = apply_feature_engineering(val)
test_processed = apply_feature_engineering(test)

X_train = train_processed[['MACD', 'Signal_Line', 'RSI']]
y_train = train_processed['Final_Signal']

X_val = val_processed[['MACD', 'Signal_Line', 'RSI']]
y_val = val_processed['Final_Signal']

X_test = test_processed[['MACD', 'Signal_Line', 'RSI']]
y_test = test_processed['Final_Signal']

In [ ]:
#Train-Test Split (Important: No Shuffle)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)



In [ ]:
#Model Building
# Logistic Regression
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train, y_train)


LogisticRegression()

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)


RandomForestClassifier()

In [ ]:
#Support Vector Machine (SVM)
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train, y_train)


SVC()

In [ ]:
# Model Evaluation
from sklearn.metrics import classification_report

for model in [lr, rf, svm]:
    preds = model.predict(X_test)
    print(model)
    print(classification_report(y_test, preds))

LogisticRegression()
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00        17
           0       1.00      1.00      1.00      7053

    accuracy                           1.00      7070
   macro avg       0.50      0.50      0.50      7070
weighted avg       1.00      1.00      1.00      7070

RandomForestClassifier()
              precision    recall  f1-score   support

          -1       1.00      0.06      0.11        17
           0       1.00      1.00      1.00      7053

    accuracy                           1.00      7070
   macro avg       1.00      0.53      0.55      7070
weighted avg       1.00      1.00      1.00      7070

SVC()
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00        17
           0       1.00      1.00      1.00      7053

    accuracy                           1.00      7070
   macro avg       0.50      0.50      0.50      7070
weighted avg       1.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

##When I look at the classification report, I can see that all three models, Logistic Regression, Random Forest, and SVC, performed perfectly for class 0 but completely failed to predict class −1. The precision, recall, and F1 score for class −1 are all 0.00, meaning the models never identified any samples belonging to that class. In contrast, class 0 has precision, recall, and F1 score of 1.00, showing that every prediction was correct. The overall accuracy of 1.00 is misleading because the dataset is heavily imbalanced: there are 7 053 samples of class 0 and only 17 of class −1. This imbalance caused the models to learn to always predict class 0, ignoring the minority class entirely. To improve this, I would need to rebalance the data, using techniques like oversampling, under sampling, or class weight adjustment, so the models can learn to recognize both classes.

In [ ]:
#7. Optimization (Basic)
rf = RandomForestClassifier(n_estimators=200, max_depth=10)
rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=200)

In [ ]:
#Cross Validation
from sklearn.model_selection import cross_val_score

scores = cross_val_score(rf, X, y, cv=5)
print(scores.mean())


0.9989532478701056


##The cross validation score of 0.9989 indicates extremely high predictive performance, but this result is misleading due to severe class imbalance in the dataset. The model achieves near perfect accuracy by consistently predicting the majority class (0), while completely failing to identify the minority class (−1). Therefore, the high cross validation score does not reflect true model generalization and highlights the need for rebalancing techniques such as oversampling, under sampling, or class weight adjustment.

##Github link https://github.com/christinengalle19-collab/Christine-Tsafong-BIA-PROJECTS.git


##Colab linkhttps: https://colab.research.google.com/drive/1McutG0IfaD6Ynt_9J3lB4IglurULMqgL?usp=sharing